
# Create or update images table

**Author:** Juliana Thomasia C. Marcos\
**Input:** Databricks images table information (e.g., name)\
**Output:** Databricks image metadata table (`TABLE_NAME`)

## Overview

This notebook creates or updates an images table in Databricks using imagery data from a specified source folder. It extracts image metadata and generates the corresponding table records through batch processing with Apache Spark.

The notebook supports both incremental updates and full regeneration through the `REGEN` parameter. In incremental mode (`REGEN=False`), it compares the source folder's image paths against existing table records to identify and process only newly added images. The resulting records are combined with existing records for images still present in the source folder. When regeneration is enabled (`REGEN=True`), all source images are reprocessed and the table is rebuilt from scratch.

# Packages import and installation

In [0]:
%pip install ..

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from functools import partial

from ag_vision.pipelines import image_processing as ip
from ag_vision.pipelines import dataframe_processing as dfp

# Variables set up

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
catalog_name = "use1_prod_artemis_catalog_3718194974443840"

In [0]:
# Images table name
table_name = dbutils.widgets.get("table_name")
TABLE_NAME = f"{catalog_name}.production.{table_name}"

In [0]:
# Set to True to generate the whole table from scratch (this will read all metadatafile).
# Set to False to only add new data (this will read only metadata files for images not in the table).
REGEN = dbutils.widgets.get("REGEN").strip().lower() == "true"
print(f"Regen is {REGEN}, if true it will recreate the whole table, if False it will append new data to the existing table.")

In [0]:
# Set the root directory variables
folder_name = dbutils.widgets.get("folder_name")
root_path = f"/Volumes/{catalog_name}/production/data/{folder_name}"

# Imagery data loading from source folder

In [0]:
# Load the root directory
# Note: Loading from the 'artemis' subfolder to make the count consistent
#glob_pattern = f"{root_path}/*/*/*/*/*/im*/**/*.{{jpg,jpeg,png,webp}}"
df = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load(root_path) \
    .select("path")

# Filter by Depth AND Regex
# Your pattern: e.g., artemis/*/*/*/*/*/*/im*/**/*
# This implies a specific number of folders between 'artemis' and the file.
img_paths_df = df.select("path").filter(
    # Ensure it's an image
    F.col("path").rlike(r"\.(jpg|jpeg|png|webp)$") & 
    # Ensure 'im' folder is in the path
    F.col("path").contains("/im") &
    # Ensure the depth matches your 13-slash pattern, this will make sure annotation images done make it in.
    (F.size(F.split(F.col("path"), "/")) >= 13)
)

# Collect
img_paths_list = [row.path.replace('dbfs:', '') for row in img_paths_df.toLocalIterator()]
print(f"Found {len(img_paths_list)} paths at the correct depth.")

In [0]:
# Load the relevant column from the current table into a spark dataframe 
old_df = spark.table(TABLE_NAME).select("file_path").toPandas()
print(f"The Old Table len is {len(old_df)}")

# Identification of newly uploaded images data

In [0]:
if not REGEN:
    s = pd.Series(img_paths_list)
    run_list = s[~s.isin(old_df['file_path'])].tolist()
else:
    run_list = img_paths_list
    
print(f"The len of the run list is. {len(run_list)}")

In [0]:
# Table schema generation.
if len(run_list) > 0:
    sample_df = ip.generate_images_table(img_list=run_list[:1],
                                        platform='db')

    sample_df['blur'] = sample_df['blur'].astype(int)
    my_schema = spark.createDataFrame(sample_df).schema.simpleString()
    df = spark.createDataFrame([(i,) for i in run_list], ["item"])

In [0]:
if len(run_list) > 0:
    img_spark_df = df.mapInPandas(
        partial(dfp.process_images_batch, sample_df=sample_df),
        schema=my_schema,
    )

In [0]:
files_sdf = spark.createDataFrame([(path,) for path in img_paths_list], ["file_path"])

# Table update with new images data

In [0]:
# 1. Check if the table exists and REGEN is False
table_exists = spark.catalog.tableExists(TABLE_NAME)

if not REGEN and table_exists:
    # Create a DataFrame from img_paths_list to avoid driver memory issues with .isin()
    old_spark_df = spark.read.table(TABLE_NAME).join(files_sdf, "file_path", "inner")
    
    # Combine (Union) the old data with the new results
    # unionByName is safer as it matches columns by name, not position
    if len(run_list) > 0:
        final_spark_df = old_spark_df.unionByName(img_spark_df)
    else:
        final_spark_df = old_spark_df
else:
    # If REGEN is True or table doesn't exist, just use the new data
    final_spark_df = img_spark_df

#2. Write the final result back to the table
final_spark_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)